In [21]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
pd.set_option('display.max_rows', None)

# <font color='yellow'>Data Cleaning</font>

In [22]:
df = pd.read_csv('Bengaluru_House_Data.csv')

In [23]:
print('data:->')
print(df.head(3))
print()
print('--------------------------------------------------------------------------------------------------')

print('datatypes:-> ')
print(df.info())
print()
print('---------------------------------------------------------------------------------------------------')


print('shape:-> ')
print(df.shape)
print()
print('---------------------------------------------------------------------------------------------------')

print('description:-> ')
print(df.describe(include='all'))
print()
print('---------------------------------------------------------------------------------------------------')

print('null count:-> ')
print(df.isnull().sum())
print()
print('---------------------------------------------------------------------------------------------------')

print('duplicate count:-> ')
print(df.duplicated().sum())
print()
print('---------------------------------------------------------------------------------------------------')

data:->
              area_type   availability                  location       size  \
0  Super built-up  Area         19-Dec  Electronic City Phase II      2 BHK   
1            Plot  Area  Ready To Move          Chikka Tirupathi  4 Bedroom   
2        Built-up  Area  Ready To Move               Uttarahalli      3 BHK   

   society total_sqft  bath  balcony   price  
0  Coomee        1056   2.0      1.0   39.07  
1  Theanmp       2600   5.0      3.0  120.00  
2      NaN       1440   2.0      3.0   62.00  

--------------------------------------------------------------------------------------------------
datatypes:-> 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   so

In [24]:
# we only want location, size, price and total_sqt 
# we will remove the 529 duplicates 
df = df[['location','total_sqft','size','price']]
df = df.drop_duplicates()
print(df.duplicated().sum())
print()
print(df.isnull().sum())

0

location       1
total_sqft     0
size          16
price          0
dtype: int64


In [25]:
df['location'].value_counts().index

Index(['Whitefield', 'Sarjapur  Road', 'Electronic City', 'Thanisandra',
       'Kanakpura Road', 'Yelahanka', 'Marathahalli', 'Raja Rajeshwari Nagar',
       'Bannerghatta Road', 'Uttarahalli',
       ...
       'Maruthi Extension', 'Okalipura', 'Old Town', 'Vasantapura main road',
       'Bapuji Layout', '1st Stage Radha Krishna Layout',
       'BEML Layout 5th stage', 'singapura paradise', 'Uvce Layout',
       'Abshot Layout'],
      dtype='object', name='location', length=1305)

In [26]:
#handling the null values:->

#location null values handling
df['location'] = df['location'].fillna(df['location'].value_counts().index[0])

#size null value handling 
   # we can do this :-> df['size'] = df['size'].fillna(df['size'].median())
   #but the size is a object so we will handle it later

print(df.isnull().sum())

location       0
total_sqft     0
size          16
price          0
dtype: int64


In [27]:
#lets handle 'size'
print(df['size'].value_counts())

size
2 BHK         4762
3 BHK         3962
4 Bedroom      814
4 BHK          555
3 Bedroom      523
1 BHK          504
2 Bedroom      302
5 Bedroom      288
6 Bedroom      191
1 Bedroom      100
8 Bedroom       84
7 Bedroom       82
5 BHK           59
9 Bedroom       46
6 BHK           30
7 BHK           17
1 RK            12
10 Bedroom      11
9 BHK            8
8 BHK            5
11 BHK           2
11 Bedroom       2
10 BHK           2
14 BHK           1
13 BHK           1
12 Bedroom       1
27 BHK           1
43 Bedroom       1
16 BHK           1
19 BHK           1
18 Bedroom       1
Name: count, dtype: int64


In [28]:
#convering df['size'] into numerical data 
df['size'] = df['size'].str.strip().str.split().str[0]
print(df['size'])

0          2
1          4
2          3
3          3
4          2
5          2
6          4
7          4
8          3
9          6
10         3
11         4
12         2
13         2
14         3
15         2
16         3
17         3
18         3
19         2
20         1
21         3
22         4
23         3
24         1
25         3
26         2
27         3
28         2
29         3
30         4
31         3
32         3
33         3
34         3
35         2
36         2
37         3
38         3
39         2
40         2
41         3
42         1
43         1
44         2
45         8
46         2
47         2
48         2
49         2
50         2
51         3
52         3
53         2
54         3
55         2
56         4
57         2
58         6
59         2
60         3
61         2
62         4
63         2
64         8
65         2
66         2
67         2
68         8
69         2
70         3
71         2
72         3
73         2
74         3
75         2
76         2

In [29]:
df['size'] = df['size'].fillna('2')
df.isnull().sum()

location      0
total_sqft    0
size          0
price         0
dtype: int64

In [30]:
#converting datatypes of size into numerical column
df['size'] = df['size'].astype(int)

In [31]:
df['total_sqft'].value_counts()

total_sqft
1200                 769
1100                 200
1500                 198
2400                 194
600                  174
1000                 164
1350                 124
1050                 110
1250                 108
1300                 104
1800                 102
1400                 101
1600                  99
900                   98
1150                  96
2000                  79
1140                  71
800                   67
1450                  67
3000                  66
1650                  63
2500                  62
950                   59
1550                  57
1700                  57
1180                  55
1080                  54
1020                  54
1125                  53
750                   52
1260                  52
1075                  50
1160                  49
1070                  47
4000                  46
1225                  46
1220                  46
2100                  45
1240                  45
1060          

In [32]:
#as we can see there are many faults in df['total_sqft']
#we will convert values like ( abc - def ) from their mean() 
# and we will convert other faults as NAN

def convert_sqft_to_num(x):
    # 1. Handle ranges like '2100 - 2850'
    tokens = str(x).split('-')
    if len(tokens) == 2:
        return (float(tokens[0].strip()) + float(tokens[1].strip())) / 2
    
    # 2. Try converting single standard numbers like '1056'
    try:
        return float(x)
    except:
        # 3. Handle unparseable strings like '34.46Sq. Meter'
        return None

df['total_sqft'] = df['total_sqft'].apply(convert_sqft_to_num)
print(df['total_sqft'].value_counts())

        

total_sqft
1200.000     769
1100.000     200
1500.000     199
2400.000     194
600.000      174
1000.000     165
1350.000     124
1050.000     110
1250.000     108
1800.000     105
1300.000     104
1400.000     101
1600.000      99
900.000       98
1150.000      96
2000.000      79
1140.000      71
800.000       69
1450.000      68
3000.000      66
1650.000      63
2500.000      63
950.000       59
1700.000      58
1550.000      57
1180.000      55
1080.000      54
1020.000      54
1125.000      53
1260.000      53
750.000       52
1075.000      51
1160.000      49
1070.000      47
1225.000      46
4000.000      46
1220.000      46
1240.000      45
2100.000      45
700.000       44
1060.000      44
1175.000      44
1210.000      43
850.000       42
1230.000      42
1320.000      40
1280.000      40
1170.000      39
1190.000      37
1750.000      37
1255.000      37
1410.000      37
1185.000      37
1025.000      34
1290.000      34
1270.000      34
1850.000      33
1310.000      33
270

In [33]:
df['total_sqft'] = df['total_sqft'].fillna(df['total_sqft'].median())

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12385 entries, 0 to 13318
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    12385 non-null  object 
 1   total_sqft  12385 non-null  float64
 2   size        12385 non-null  int64  
 3   price       12385 non-null  float64
dtypes: float64(2), int64(1), object(1)
memory usage: 483.8+ KB


In [35]:
#handling location column 
df['location'].value_counts()

location
Whitefield                                            507
Sarjapur  Road                                        360
Electronic City                                       272
Thanisandra                                           224
Kanakpura Road                                        216
Yelahanka                                             208
Marathahalli                                          164
Raja Rajeshwari Nagar                                 153
Bannerghatta Road                                     149
Uttarahalli                                           143
Hebbal                                                141
Hennur Road                                           140
7th Phase JP Nagar                                    125
Electronic City Phase II                              106
Rajaji Nagar                                          106
Bellandur                                              93
KR Puram                                               91
Elect

In [36]:
#there are too many categories in location so we will only take those categories whose value_counts is >=10
real_location_list = df['location'].value_counts()
real_location_list[real_location_list >=10 ]

location
Whitefield                     507
Sarjapur  Road                 360
Electronic City                272
Thanisandra                    224
Kanakpura Road                 216
Yelahanka                      208
Marathahalli                   164
Raja Rajeshwari Nagar          153
Bannerghatta Road              149
Uttarahalli                    143
Hebbal                         141
Hennur Road                    140
7th Phase JP Nagar             125
Electronic City Phase II       106
Rajaji Nagar                   106
Bellandur                       93
KR Puram                        91
Electronics City Phase 1        88
Hoodi                           88
Yeshwanthpur                    83
Haralur Road                    83
Sarjapur                        82
Kasavanhalli                    79
Harlur                          74
Kengeri                         72
Hormavu                         72
Begur Road                      72
JP Nagar                        71
Ramamurthy 

In [37]:
top_locations = real_location_list[real_location_list >=10].index

def handling_location(text):
    if text in top_locations:
        return text
    else:
        return'Other'

df['location'] = df['location'].apply(handling_location)
df['location'].value_counts()

location
Other                          2782
Whitefield                      507
Sarjapur  Road                  360
Electronic City                 272
Thanisandra                     224
Kanakpura Road                  216
Yelahanka                       208
Marathahalli                    164
Raja Rajeshwari Nagar           153
Bannerghatta Road               149
Uttarahalli                     143
Hebbal                          141
Hennur Road                     140
7th Phase JP Nagar              125
Electronic City Phase II        106
Rajaji Nagar                    106
Bellandur                        93
KR Puram                         91
Hoodi                            88
Electronics City Phase 1         88
Yeshwanthpur                     83
Haralur Road                     83
Sarjapur                         82
Kasavanhalli                     79
Harlur                           74
Kengeri                          72
Hormavu                          72
Begur Road         

In [38]:
len(df['location'].value_counts())

247

In [39]:
print(df['location'].unique())

['Electronic City Phase II' 'Chikka Tirupathi' 'Uttarahalli'
 'Lingadheeranahalli' 'Kothanur' 'Whitefield' 'Old Airport Road'
 'Rajaji Nagar' 'Marathahalli' 'Other' '7th Phase JP Nagar' 'Gottigere'
 'Sarjapur' 'Mysore Road' 'Bisuvanahalli' 'Raja Rajeshwari Nagar'
 'Kengeri' 'Binny Pete' 'Thanisandra' 'Bellandur' 'Electronic City'
 'Ramagondanahalli' 'Yelahanka' 'Hebbal' 'Kasturi Nagar' 'Kanakpura Road'
 'Electronics City Phase 1' 'Kundalahalli' 'Chikkalasandra'
 'Murugeshpalya' 'Sarjapur  Road' 'Ganga Nagar' 'HSR Layout'
 'Doddathoguru' 'KR Puram' 'Bhoganhalli' 'Lakshminarayana Pura'
 'Begur Road' 'Devanahalli' 'Varthur' 'Bommanahalli' 'Gunjur'
 'Hegde Nagar' 'Haralur Road' 'Hennur Road' 'Kothannur' 'Kalena Agrahara'
 'Kaval Byrasandra' 'ISRO Layout' 'Garudachar Palya' 'EPIP Zone'
 'Dasanapura' 'Kasavanhalli' 'Sanjay nagar' 'Domlur'
 'Sarjapura - Attibele Road' 'Yeshwanthpur' 'Chandapura' 'Nagarbhavi'
 'Ramamurthy Nagar' 'Malleshwaram' 'Akshaya Nagar' 'Shampura' 'Kadugodi'
 'LB Shastri

In [83]:
#final checking 
print('data:->')
print(df.head(3))
print()
print('--------------------------------------------------------------------------------------------------')

print('datatypes:-> ')
print(df.info())
print()
print('---------------------------------------------------------------------------------------------------')


print('shape:-> ')
print(df.shape)
print()
print('---------------------------------------------------------------------------------------------------')

print('description:-> ')
print(df.describe(include='all'))
print()
print('---------------------------------------------------------------------------------------------------')

print('null count:-> ')
print(df.isnull().sum())
print()
print('---------------------------------------------------------------------------------------------------')

print('duplicate count:-> ')
print(df.duplicated().sum())
print()
print('---------------------------------------------------------------------------------------------------')

data:->
                   location  total_sqft  size   price
0  Electronic City Phase II      1056.0     2   39.07
1          Chikka Tirupathi      2600.0     4  120.00
2               Uttarahalli      1440.0     3   62.00

--------------------------------------------------------------------------------------------------
datatypes:-> 
<class 'pandas.core.frame.DataFrame'>
Index: 12385 entries, 0 to 13318
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    12385 non-null  object 
 1   total_sqft  12385 non-null  float64
 2   size        12385 non-null  int64  
 3   price       12385 non-null  float64
dtypes: float64(2), int64(1), object(1)
memory usage: 483.8+ KB
None

---------------------------------------------------------------------------------------------------
shape:-> 
(12385, 4)

---------------------------------------------------------------------------------------------------
description:-> 
     

In [84]:
df = df.drop_duplicates()

In [85]:
#data cleaning completed
#saving the cleaned data 
df.to_csv('cleaned_dataset.csv',index=False)
print('dataset saved successfully:)')

dataset saved successfully:)
